# Nghiên Cứu: Hệ Gợi Ý Sản Phẩm Điện Tử (Cross-Domain & Big Data)
**Mục tiêu:** Xây dựng hệ thống gợi ý chuyển giao tri thức từ Amazon (Tiếng Anh) sang Web VN (Tiếng Việt) sử dụng Đồ thị dị thể (HGNN) và công nghệ Big Data (Spark, Vector DB).

In [ ]:
# Cài đặt Apache Spark cho xử lý dữ liệu lớn
!pip install pyspark==3.4.1
!pip install findspark

# Cài đặt PyTorch Geometric (PyG) cho Graph Neural Networks
!pip install torch_geometric
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.1.0+cu118.html

# Cài đặt thư viện tìm kiếm vector (Mô phỏng cho Milvus/Vector DB)
!pip install faiss-cpu

# Thư viện NLP đa ngữ (mBERT/XLM-R)
!pip install transformers

## Giai đoạn 2: Xây dựng Pipeline Xử lý Dữ liệu Lớn (Big Data Processing)
[cite_start]Xử lý dữ liệu Amazon Reviews và dữ liệu Web VN mà không bị tràn RAM bằng Apache Spark[cite: 17]. [cite_start]Công việc bao gồm làm sạch text, xử lý missing values, và chuẩn bị dữ liệu dạng đồ thị[cite: 21].

In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, regexp_replace

# Khởi tạo Spark Session với bộ nhớ được tinh chỉnh
spark = SparkSession.builder \
    .appName("CrossDomainRecSys_DataPrep") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()

print("Spark version:", spark.version)

# ---------------------------------------------------------
# GIẢ LẬP ĐỌC DỮ LIỆU
# Tải bộ dữ liệu Amazon Product Data (category Electronics) [cite: 19]
# Thu thập/Chuẩn bị bộ dữ liệu Web VN [cite: 20]
# ---------------------------------------------------------
# df_amazon = spark.read.json("path/to/amazon_electronics.json")
# df_vn = spark.read.parquet("path/to/vn_ecommerce.parquet") [cite: 26]

# Ví dụ hàm làm sạch text cơ bản bằng Spark [cite: 21]
def clean_text_spark(df, column_name):
    return df.withColumn(
        f"{column_name}_cleaned",
        lower(regexp_replace(col(column_name), "[^a-zA-Z0-9\\s]", ""))
    )

# df_amazon_clean = clean_text_spark(df_amazon, "reviewText")
# df_amazon_clean.show(5)

## Giai đoạn 3: Thiết kế Kiến trúc HGNN & Pre-training trên Amazon
[cite_start]Xây dựng mô hình Heterogeneous Graph Neural Networks (HGNN) kết nối User - Item - Brand - Specs[cite: 31]. Mạng này sẽ học được biểu diễn (embeddings) cho người dùng và sản phẩm.

In [ ]:
import torch
import torch_geometric.transforms as T
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, GATConv, SAGEConv

# 1. Định nghĩa cấu trúc Đồ thị dị thể (HeteroData)
data = HeteroData()

# Thêm node (Giả lập số lượng node)
data['user'].x = torch.randn(1000, 128) # 1000 users, feature dim = 128
data['item'].x = torch.randn(5000, 128) # 5000 items (Điện tử)
data['brand'].x = torch.randn(100, 64)  # 100 brands

# Thêm edge (Cạnh tương tác)
data['user', 'clicks', 'item'].edge_index = torch.randint(0, 500, (2, 10000))
data['item', 'belongs_to', 'brand'].edge_index = torch.randint(0, 100, (2, 5000))

# 2. Định nghĩa Mô hình HGNN [cite: 31, 35]
class CrossDomainHGNN(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = HeteroConv({
            ('user', 'clicks', 'item'): SAGEConv((-1, -1), hidden_channels),
            ('item', 'belongs_to', 'brand'): GATConv((-1, -1), hidden_channels),
        }, aggr='sum')

        self.conv2 = HeteroConv({
            ('user', 'clicks', 'item'): SAGEConv((-1, -1), out_channels),
            ('item', 'belongs_to', 'brand'): GATConv((-1, -1), out_channels),
        }, aggr='sum')

    def forward(self, x_dict, edge_index_dict):
        x_dict = self.conv1(x_dict, edge_index_dict)
        x_dict = {key: x.relu() for key, x in x_dict.items()}
        x_dict = self.conv2(x_dict, edge_index_dict)
        return x_dict

model = CrossDomainHGNN(hidden_channels=64, out_channels=32)
print(model)

# Lưu ý: Trong thực tế bạn cần implement Contrastive Loss ở bước huấn luyện[cite: 32].

## Giai đoạn 4: Domain Adaptation & Fine-tuning trên Web VN
Chuyển giao tri thức từ Amazon sang Web VN. [cite_start]Quá trình này sẽ đóng băng (Freeze) một số lớp biểu diễn chung và chỉ Fine-tune các lớp biểu diễn ngôn ngữ/hành vi người dùng VN để tránh Negative Transfer[cite: 39, 41, 42].

In [ ]:
import torch.nn as nn
import torch.optim as optim

# Đóng băng các lớp biểu diễn cấu hình/đặc trưng chung [cite: 42]
for param in model.conv1.parameters():
    param.requires_grad = False

# Chỉ fine-tune lớp cuối cho dữ liệu VN [cite: 42]
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

def train_finetune(model, vn_data):
    model.train()
    optimizer.zero_grad()
    # Đầu ra vector nhúng của mô hình
    out_dict = model(vn_data.x_dict, vn_data.edge_index_dict)

    # Giả lập hàm loss (Trong thực tế dùng BPR Loss hoặc Contrastive Loss)
    # loss = contrastive_loss(out_dict['user'], out_dict['item'], positive_edges, negative_edges)
    loss = torch.tensor(0.5, requires_grad=True) # Placeholder

    loss.backward()
    optimizer.step()
    return loss.item()

# Giả lập chạy epoch
# for epoch in range(10):
#     loss = train_finetune(model, vn_graph_data)
#     print(f"Epoch {epoch} | Loss: {loss}")

## Giai đoạn 5: Tối ưu Kiến trúc Hệ thống (Two-Stage Serving)
[cite_start]Thiết kế hệ thống Retrieval (Truy xuất thô bằng thuật toán ANN tìm kiếm xấp xỉ) và Ranking (Xếp hạng tinh) đáp ứng độ trễ thấp[cite: 45, 47, 51].

In [ ]:
import faiss
import numpy as np

# Giả sử sau khi train, ta có được ma trận vector nhúng của 5000 sản phẩm (Item Embeddings)
# Dimension = 32 (Từ out_channels của model)
item_embeddings = np.random.random((5000, 32)).astype('float32')
user_query_vector = np.random.random((1, 32)).astype('float32')

# 1. Retrieval Stage: Dùng FAISS để tạo index tìm kiếm cực nhanh (ANN Search) [cite: 47, 51]
dimension = 32
index = faiss.IndexFlatIP(dimension) # Inner Product (Tương đương Cosine Similarity nếu vector đã normalize)
faiss.normalize_L2(item_embeddings)
index.add(item_embeddings)

# Tìm top 100 sản phẩm thô cho user (Retrieval)
faiss.normalize_L2(user_query_vector)
k_retrieval = 100
distances, indices = index.search(user_query_vector, k_retrieval)

print(f"Top {k_retrieval} candidate item IDs (Retrieval Stage):", indices[0][:10], "...")

# 2. Ranking Stage (Xếp hạng tinh) [cite: 47]
# Đưa top 100 candidate này qua một model nhẹ (LightGBM/MLP) kèm theo real-time context
# để chọn ra Top 10 cuối cùng hiển thị cho người dùng.

## Giai đoạn 6: Đánh giá Benchmark
[cite_start]Chứng minh tính hiệu quả qua các độ đo Machine Learning: NDCG@10, Hit Ratio@10, MRR, đặc biệt trên tập Cold-start[cite: 56, 57]. [cite_start]Đồng thời đánh giá chỉ số hệ thống như QPS và Latency[cite: 58].

In [ ]:
import math

def get_ndcg_at_k(recommended_list, ground_truth_list, k=10):
    """Tính toán NDCG@10 [cite: 57]"""
    dcg = 0.0
    idcg = 0.0
    for i, item in enumerate(recommended_list[:k]):
        if item in ground_truth_list:
            dcg += 1.0 / math.log2(i + 2)
    for i in range(min(k, len(ground_truth_list))):
        idcg += 1.0 / math.log2(i + 2)

    return dcg / idcg if idcg > 0 else 0.0

def get_hit_ratio_at_k(recommended_list, ground_truth_list, k=10):
    """Tính toán Hit Ratio@10 [cite: 57]"""
    for item in recommended_list[:k]:
        if item in ground_truth_list:
            return 1.0
    return 0.0

# Giả lập đánh giá
recommendations = [12, 45, 88, 102, 5, 19, 21, 55, 9, 3] # Top 10 model trả về
actual_clicks = [88, 9] # Sản phẩm user thực sự click

print(f"NDCG@10: {get_ndcg_at_k(recommendations, actual_clicks):.4f}")
print(f"Hit Ratio@10: {get_hit_ratio_at_k(recommendations, actual_clicks):.4f}")